# tables_vizualization — acompanhamento das tabelas do pipeline

Notebook da pasta `exploration`: consultas de **leitura** contra as tabelas já gravadas no catálogo, para conferir o resultado após cada execução do pipeline — shape por edição em todas as camadas (bronze → gold), a linha do ledger (`pipeline_manifest`, contrato de execução) e o histórico Delta das tabelas silver (`DESCRIBE HISTORY`, para conferir o `userMetadata` das gravações do upsert). Não grava nada.

Convenção da pasta: inspeção de tabela persistida mora aqui; o notebook de pipeline fica só com células de produção. O que depende de variável intermediária de sessão (DataFrame ainda não gravado) continua no notebook do pipeline.

In [0]:
%run ../environment/env_variables

In [0]:
# Tabelas do pipeline (todas as camadas — nomes centralizados no env_variables)
bronze_tbl_dict = f"{catalog_name}.{bronze_schema}.munic_raw_dict"
bronze_tbl_info = f"{catalog_name}.{bronze_schema}.munic_raw_info_data"
bronze_tbl_gov  = f"{catalog_name}.{bronze_schema}.munic_raw_gov_data"

silver_tbl_data = f"{catalog_name}.{silver_schema}.munic_clean_data"
silver_tbl_dict = f"{catalog_name}.{silver_schema}.munic_clean_dict"

gold_dim_municipio   = f"{catalog_name}.{gold_schema}.dim_municipio"
gold_dim_indicador   = f"{catalog_name}.{gold_schema}.dim_indicador"
gold_fact_resposta   = f"{catalog_name}.{gold_schema}.fact_resposta"
gold_fact_imdm       = f"{catalog_name}.{gold_schema}.fact_imdm_municipio"

for v in (bronze_tbl_dict, bronze_tbl_info, bronze_tbl_gov, silver_tbl_data, silver_tbl_dict,
          gold_dim_municipio, gold_dim_indicador, gold_fact_resposta, gold_fact_imdm, manifest_tbl):
    print(v)

## Shape por edição (todas as camadas)

Linhas por `ano_edicao` em cada tabela principal — a primeira conferência pós-run: cada camada deve carregar as duas edições, e os totais devem bater com os declarados no ledger.

In [0]:
tabelas_shape = {
    "bronze munic_raw_info_data": bronze_tbl_info,
    "bronze munic_raw_gov_data":  bronze_tbl_gov,
    "bronze munic_raw_dict":      bronze_tbl_dict,
    "silver munic_clean_data":    silver_tbl_data,
    "silver munic_clean_dict":    silver_tbl_dict,
    "gold dim_municipio":         gold_dim_municipio,
    "gold dim_indicador":         gold_dim_indicador,
    "gold fact_resposta":         gold_fact_resposta,
    "gold fact_imdm_municipio":   gold_fact_imdm,
}

for nome, tabela in tabelas_shape.items():
    por_edicao = spark.sql(
        f"SELECT ano_edicao, COUNT(*) AS linhas FROM {tabela} GROUP BY ano_edicao ORDER BY ano_edicao"
    ).collect()
    print(f"{nome}: " + ", ".join(f"{r['ano_edicao']} = {r['linhas']}" for r in por_edicao))

## Ledger (`pipeline_manifest`) — contrato de execução

Uma linha por etapa × edição: linhas, indicadores com dados, recusa preservada e o SHA-256 do insumo que a etapa consumiu (`docs/arquitetura/pipeline_manifest.md`).

In [0]:
display(spark.sql(f"SELECT * FROM {manifest_tbl} ORDER BY etapa, ano_edicao"))

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {manifest_tbl}"))

## Bronze — SELECT * e histórico por tabela

Para cada tabela da bronze: algumas linhas da tabela (SELECT *) e, logo abaixo, o histórico Delta da escrita (DESCRIBE HISTORY).

In [0]:
display(spark.sql(f"SELECT * FROM {bronze_tbl_info}"))

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {bronze_tbl_info}"))

In [0]:
display(spark.sql(f"SELECT * FROM {bronze_tbl_gov}"))

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {bronze_tbl_gov}"))

In [0]:
display(spark.sql(f"SELECT * FROM {bronze_tbl_dict}"))

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {bronze_tbl_dict}"))

## Silver — SELECT * e histórico por tabela

Para cada tabela da silver: algumas linhas da tabela (SELECT *) e, logo abaixo, o histórico Delta (DESCRIBE HISTORY) — com o `userMetadata`, a descrição da operação registrada pelo `upsert_delta_table` (auditoria das gravações).

In [0]:
display(spark.sql(f"SELECT * FROM {silver_tbl_data}"))

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {silver_tbl_data}"))

In [0]:
display(spark.sql(f"SELECT * FROM {silver_tbl_dict}"))

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {silver_tbl_dict}"))

## Gold — SELECT * e histórico por tabela

Para cada tabela dimensional/fato da gold: umas linhas da tabela (SELECT *) e, logo abaixo, o histórico Delta (DESCRIBE HISTORY).

In [0]:
display(spark.sql(f"SELECT * FROM {gold_dim_municipio}"))

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {gold_dim_municipio}"))

In [0]:
display(spark.sql(f"SELECT * FROM {gold_dim_indicador}"))

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {gold_dim_indicador}"))

In [0]:
display(spark.sql(f"SELECT * FROM {gold_fact_resposta}"))

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {gold_fact_resposta}"))

In [0]:
display(spark.sql(f"SELECT * FROM {gold_fact_imdm}"))

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {gold_fact_imdm}"))

## Gold — SELECT * e histórico por tabela

Para cada tabela dimensional/fato da gold: algumas linhas da tabela (SELECT *) e, logo abaixo, o histórico Delta (DESCRIBE HISTORY).

In [0]:
display(spark.sql(f"SELECT * FROM {gold_dim_municipio}"))

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {gold_dim_municipio}"))

In [0]:
display(spark.sql(f"SELECT * FROM {gold_dim_indicador}"))

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {gold_dim_indicador}"))

In [0]:
display(spark.sql(f"SELECT * FROM {gold_fact_resposta}"))

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {gold_fact_resposta}"))

In [0]:
display(spark.sql(f"SELECT * FROM {gold_fact_imdm}"))

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {gold_fact_imdm}"))